# Fine-Tuning Qwen3-4B-Thinking

This notebook provides a complete pipeline to fine-tune the **Qwen3-4B-Thinking** model using **QLoRA** (4-bit quantization + LoRA adapters).

## 1. Environment Setup

We need `peft` for LoRA, `trl` for the SFT (Supervised Fine-Tuning) trainer, and `bitsandbytes` for quantization.

In [4]:
# Restore pip if missing and install fine-tuning dependencies
!.venv/bin/python -m ensurepip --upgrade
!.venv/bin/python -m pip install -U peft trl bitsandbytes datasets accelerate

Looking in links: /tmp/tmp4qmeem0d


  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/680.7 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 59.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/529.0 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.9 MB/s  0:00:00
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/48.9 MB ? eta -:--:--

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 8/9 [peft]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 8/9 [peft]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 8/9 [peft]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [peft]



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /home/acojocaru/151B_SP26_Competition/.venv/bin/python -m pip install --upgrade pip


In [5]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Configuration
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/final-odyssey-math-cleaned.jsonl"
OUTPUT_DIR = "./results/qwen3_math_lora"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 2. Load Dataset

We load the local JSONL dataset and format it for instruction tuning. 
The model expects a conversation-like format or a prompt-response pair.

In [6]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_instruction(sample):
    """
    Format the dataset into a conversation format matching the inference prompt.
    """
    # Standard competition prompt
    system_prompt = "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."
    
    # Extract reasoning and answer
    reasoning = sample.get("reasoning", "")
    answer = sample.get("answer", "")
    if isinstance(answer, list):
        answer = ", ".join(map(str, answer))
    
    # Construction of the full response
    # Qwen3-Thinking usually generates reasoning followed by the answer.
    full_response = f"{reasoning}\n\n\\boxed{{{answer}}}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sample["question"]},
        {"role": "assistant", "content": full_response}
    ]
    
    return {"messages": messages}

dataset = dataset.map(format_instruction)
print(f"Sample formatted message:\n{dataset[0]['messages']}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/387 [00:00<?, ? examples/s]

Sample formatted message:
[{'role': 'system', 'content': 'You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}.'}, {'role': 'user', 'content': 'Let $S=\\left\\{ 1,2,\\cdots 2024 \\right\\}$, if the set of any $n$ pairwise prime numbers in $S$ has at least one prime number, the minimum value of $n$ is \\underline{\\hspace{2cm}}.'}, {'role': 'assistant', 'content': 'Taking the 15 numbers 1, 22, 32, ..., 432 violates the condition. Furthermore, since $S$ does not contain any non-prime numbers with a minimum prime factor of at least 47, there are only 14 types of non-prime numbers in $S$, excluding 1. Applying the Pigeonhole Principle, we conclude that $n=16$.\n\n\\boxed{16}'}]


## 3. Model Initialization (QLoRA)

We load the model in 4-bit to save memory and prepare it for LoRA training.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/acojocaru/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## 4. Training

We use the `SFTTrainer` which handles the chat template automatically if the dataset contains a `messages` column.

In [10]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    num_train_epochs=3, 
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

NameError: name 'SFTConfig' is not defined

## 5. Save and Test

Save the adapter and run a quick test.

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "final_adapter"))
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

In [ ]:
# Quick test
question = "What is the sum of the first 10 positive even numbers?"
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."},
        {"role": "user", "content": question}
    ], 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    output_tokens = model.generate(**inputs, max_new_tokens=512, temperature=0.1)

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))